In [43]:
import torch
from torch import nn
import math
import copy
import numpy as np
import sentencepiece as spm

seed=42
torch.manual_seed(seed)
np.random.seed(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)
     

cuda


In [44]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [45]:
def attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, v)

class MultiheadAttention(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        assert demb % h == 0
        self.d_k = demb // h
        self.h = h
        self.demb = demb
        self.linears = clones(nn.Linear(demb, demb), 4)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        # if mask is not None:
        #     mask = mask.unsqueeze(1) #adds a dimension to mask at second position for head axis
        nbatches = query.size(0)

        query, key, value = [linear(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) for linear, x in zip(self.linears, (query, key, value))]
        """ 
            this results in query, key, value of shape (batch_size, num_heads, seq_len, d_k) 
            taking dv=dk=demb//h
        """

        """
            contiguous function maintains continuty in memory space even after transpose, or else it might show an error with view function is used to reshape the tensor to the desired shape directly from memory
        """
        x = attention(query, key, value, mask=mask, dropout=self.dropout) #type:ignore
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.demb)
        return self.linears[-1](x)

In [46]:
np.random.seed(42)

x = np.random.randn(3, 4)

mha = MultiheadAttention(demb=4, h=2)

X = torch.tensor(x, dtype=torch.float32).unsqueeze(0)

print(X)  # (1, 3, 4)

mask = np.tril(np.ones((3, 3)))
mask = torch.tensor(mask, dtype=torch.float32).view(1, 3, 3)

print(mask.shape)

out = mha.forward(X, X, X, mask)
print(out.shape)

out

tensor([[[ 0.4967, -0.1383,  0.6477,  1.5230],
         [-0.2342, -0.2341,  1.5792,  0.7674],
         [-0.4695,  0.5426, -0.4634, -0.4657]]])
torch.Size([1, 3, 3])
torch.Size([1, 3, 4])


tensor([[[ 1.1252, -0.0422,  0.4641,  0.7140],
         [ 0.8931, -0.0247,  0.3729,  0.5624],
         [ 0.7797,  0.1253, -0.1597,  0.2915]]], grad_fn=<ViewBackward0>)

In [47]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

In [48]:
import torch.nn.functional as F

"""
    SwiGLU(x, W, V) = [(xW + b) ⊙ σ(xW + b)] ⊙ (xV + c)
    FFN_SwiGLU(x) = SwiGLU(x, W, V) · W_out + b_out

    W, V : demb x dff
    W_out : dff x demb
    b,c : dff
    b_out : demb
    ⊙ - element-wise multiplication (Hadamard product)
"""

class SwiGLU(nn.Module):
    def __init__(self, demb):
        super().__init__()
        dff = int(demb * 8 / 3)
        self.w_up = nn.Linear(demb, dff)
        self.w_gate = nn.Linear(demb, dff)
        self.w_down = nn.Linear(dff, demb)

        # often kaiming works best in cases of swish or silu, though we are using sigmoid so we have to experiment to choose betweeen kaiming and xavier initialization

        nn.init.kaiming_uniform_(self.w_gate.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_up.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_down.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.w_down(F.silu(self.w_up(x)) * self.w_gate(x))

In [49]:
class EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MultiheadAttention(config.demb, config.h, config.dropout)
        """
            the older tranformers used ff networks with linear layers and ReLU activation, but newer ones use SwiGLU which is a gated activation function that can capture more complex relationships in the data
        """
        self.ff = SwiGLU(config.demb)
        self.dropout = nn.Dropout(p=config.dropout)
        self.norm = nn.RMSNorm(config.demb)

    def forward(self, x, mask=None):
        attn_output = self.attn.forward(x, x, x, mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm(x)

        ff_output = self.ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm(x)

        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_embed = config.demb
        self.tok_embed = nn.Embedding(config.encoder_vocab_size, config.demb) 
        self.pos_embed = nn.Parameter(torch.zeros(1, config.max_seq_len, config.demb).to(DEVICE)) 
        self.encoder_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config.N_encoder)])
        self.dropout = nn.Dropout(config.dropout)
        self.norm = nn.RMSNorm(config.demb)

    def forward(self, x, mask=None):
        x = self.tok_embed(x)
        x_pos = self.pos_embed[:, :x.size(1), :]
        x = self.dropout(x + x_pos)
        for layer in self.encoder_blocks:
            x = layer(x, mask)
        return self.norm(x)

In [51]:
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = MultiheadAttention(config.demb, config.h, config.dropout)
        self.cross_attn = MultiheadAttention(config.demb, config.h, config.dropout)
        self.ff = SwiGLU(config.demb)
        self.dropout = nn.Dropout(p=config.dropout)
        self.norm = nn.RMSNorm(config.demb)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        self_attn_output = self.self_attn.forward(x, x, x, mask=tgt_mask)
        x = x + self.dropout(self_attn_output)
        x = self.norm(x)

        cross_attn_output = self.cross_attn.forward(x, enc_output, enc_output, mask=src_mask)
        x = x + self.dropout(cross_attn_output)
        x = self.norm(x)

        ff_output = self.ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm(x)

        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_embed = nn.Embedding(config.decoder_vocab_size, config.demb)
        self.pos_embed = nn.Parameter(torch.zeros(1, config.max_seq_len, config.demb).to(DEVICE))
        self.layers = clones(DecoderBlock(config), config.N_decoder)
        self.norm = nn.RMSNorm(config.demb)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        x = self.tok_embed(x)
        x_pos = self.pos_embed[:, :x.size(1), :]
        x = self.dropout(x + x_pos)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask=src_mask, tgt_mask=tgt_mask)
        return self.norm(x)

In [53]:
from pathlib import Path
import urllib.request
import gzip
import shutil

BASE = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw"
FILES = [
    "train.de.gz", "train.en.gz",
    "val.de.gz", "val.en.gz",
    "test_2016_flickr.de.gz", "test_2016_flickr.en.gz",
]

out = Path("multi30k_de_en")
out.mkdir(exist_ok=True)

for name in FILES:
    gz_path = out / name
    txt_path = out / name.replace("test_2016_flickr", "test").replace(".gz", "")
    url = f"{BASE}/{name}"
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, "rb") as f_in, open(txt_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Done. Files created:")
for p in sorted(out.glob("*.de")) + sorted(out.glob("*.en")):
    print(p)


Done. Files created:
multi30k_de_en\test.de
multi30k_de_en\train.de
multi30k_de_en\val.de
multi30k_de_en\test.en
multi30k_de_en\train.en
multi30k_de_en\val.en


In [54]:
import torch

from torchtext.data import Field, Example, Dataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


SRC = Field(
    tokenize="spacy",
    tokenizer_language="de_core_news_sm",
    init_token="<sos>",
    eos_token="<eos>",
    lower=True
)

TRG = Field(
    tokenize="spacy",
    tokenizer_language="en_core_web_sm",
    init_token="<sos>",
    eos_token="<eos>",
    lower=True
)


def load_translation_dataset(src_path, trg_path, src_field, trg_field):
    fields = [("src", src_field), ("trg", trg_field)]
    examples = []

    with open(src_path, encoding="utf-8") as src_file, open(trg_path, encoding="utf-8") as trg_file:
        for src_line, trg_line in zip(src_file, trg_file):
            src_line = src_line.strip()
            trg_line = trg_line.strip()

            if src_line and trg_line:
                example = Example.fromlist(
                    [src_line, trg_line],
                    fields
                )
                examples.append(example)

    return Dataset(examples, fields)


train_data = load_translation_dataset(
    "multi30k_de_en/train.de",
    "multi30k_de_en/train.en",
    SRC,
    TRG
)

valid_data = load_translation_dataset(
    "multi30k_de_en/val.de",
    "multi30k_de_en/val.en",
    SRC,
    TRG
)

test_data = load_translation_dataset(
    "multi30k_de_en/test.de",
    "multi30k_de_en/test.en",
    SRC,
    TRG
)

In [55]:
print(vars(train_data.examples[1])) #type:ignore

{'src': ['mehrere', 'männer', 'mit', 'schutzhelmen', 'bedienen', 'ein', 'antriebsradsystem', '.'], 'trg': ['several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']}


In [56]:
en_vocab_size = 8200
de_vocab_size = 10000
vocab_sizes = {"en": en_vocab_size, "de": de_vocab_size}

spm.SentencePieceTrainer.Train\
(f'--input=multi30k_de_en/train.de --model_prefix=Multi30k_de --user_defined_symbols= --vocab_size={de_vocab_size}')
spm.SentencePieceTrainer.Train\
(f'--input=multi30k_de_en/train.en --model_prefix=Multi30k_en --user_defined_symbols= --vocab_size={en_vocab_size}')

# make SentencePieceProcessor instances and load the model files
de_sp = spm.SentencePieceProcessor()
de_sp.Load('Multi30k_de.model')
en_sp = spm.SentencePieceProcessor()
en_sp.Load('Multi30k_en.model')

tokenizers = {"en": en_sp.EncodeAsIds, "de": de_sp.EncodeAsIds}
detokenizers = {"en": en_sp.DecodeIds, "de": de_sp.DecodeIds}

# encode: text => id
print(en_sp.EncodeAsPieces('This is a test'))
print(en_sp.EncodeAsIds('This is a test'))

# decode: id => text
print(en_sp.DecodePieces(['▁This', '▁is', '▁a', '▁t', 'est']))
print(en_sp.DecodeIds([301, 257, 9, 3, 2394]))

['▁Th', 'is', '▁is', '▁a', '▁test']
[301, 257, 9, 3, 2394]
▁This is a test
This is a test


In [57]:
print([en_sp.id_to_piece([c]) for c in range(10)]) #type:ignore
print([de_sp.id_to_piece([c]) for c in range(10)]) #type:ignore

print(valid_data[0].trg)

UNK, BOS, EOS, PAD = 0, 1, 2, 3

[['<unk>'], ['<s>'], ['</s>'], ['▁a'], ['.'], ['▁A'], ['▁in'], ['▁the'], ['▁on'], ['▁is']]
[['<unk>'], ['<s>'], ['</s>'], ['.'], ['▁eine'], ['▁Ein'], ['m'], ['▁in'], ['▁mit'], [',']]
['a', 'group', 'of', 'men', 'are', 'loading', 'cotton', 'onto', 'a', 'truck']


In [58]:
train_set = [
    (" ".join(ex.src), " ".join(ex.trg))
    for ex in train_data
    if len(ex.src) > 0 and len(ex.trg) > 0
]

valid_set = [
    (" ".join(ex.src), " ".join(ex.trg)) 
    for ex in valid_data
    if len(ex.src) > 0 and len(ex.trg) > 0
]

test_set = [
    (" ".join(ex.src), " ".join(ex.trg))
    for ex in test_data
    if len(ex.src) > 0 and len(ex.trg) > 0
]

print(len(train_set), len(valid_set), len(test_set))
for i in range(10):
   print(train_set[i])

29000 1014 1000
('zwei junge weiße männer sind im freien in der nähe vieler büsche .', 'two young , white males are outside near many bushes .')
('mehrere männer mit schutzhelmen bedienen ein antriebsradsystem .', 'several men in hard hats are operating a giant pulley system .')
('ein kleines mädchen klettert in ein spielhaus aus holz .', 'a little girl climbing into a wooden playhouse .')
('ein mann in einem blauen hemd steht auf einer leiter und putzt ein fenster .', 'a man in a blue shirt is standing on a ladder cleaning a window .')
('zwei männer stehen am herd und bereiten essen zu .', 'two men are at the stove preparing food .')
('ein mann in grün hält eine gitarre , während der andere mann sein hemd ansieht .', 'a man in green holds a guitar while the other man observes his shirt .')
('ein mann lächelt einen ausgestopften löwen an .', 'a man is smiling at a stuffed lion')
('ein schickes mädchen spricht mit dem handy während sie langsam die straße entlangschwebt .', 'a trendy gir

In [67]:
max_seq_len = 50
def tokenize_dataset(dataset):
    return [
        (
            torch.tensor(
                [BOS] + tokenizers["en"](src_text)[:max_seq_len - 2] + [EOS],
                dtype=torch.long
            ),
            torch.tensor(
                [BOS] + tokenizers["de"](trg_text)[:max_seq_len - 2] + [EOS],
                dtype=torch.long
            )
        )
        for src_text, trg_text in dataset
    ]
          
train_tokenized = tokenize_dataset(train_set)
valid_tokenized = tokenize_dataset(valid_set)
test_tokenized  = tokenize_dataset(test_set)


class TranslationDataset(torch.utils.data.Dataset):
    'create a dataset for torch.utils.data.DataLoader() '
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def pad_sequence(batch):
    'collate function for padding sentences such that all \
    the sentences in the batch have the same length'
    src_seqs  = [src for src, trg in batch]
    trg_seqs  = [trg for src, trg in batch]
    src_padded = torch.nn.utils.rnn.pad_sequence(src_seqs,
                                batch_first=True, padding_value = PAD)
    trg_padded = torch.nn.utils.rnn.pad_sequence(trg_seqs,
                                batch_first=True, padding_value = PAD)
    return src_padded, trg_padded

print(train_tokenized[0])


(tensor([   1,   16, 3792, 4283, 1075, 4467, 8042, 1388, 1075,    0,   46,  656,
           0,  194, 7734, 1616, 1552,   16, 2349, 5741, 1995,  194,    6,  304,
         221, 1468,    0, 2552, 6318, 3408,  518,    0,   14, 6786,   16,    4,
           2]), tensor([   1,   21,   77, 1005,  437,   21, 5894,  293,   21,    9, 1417, 4820,
        1013, 6027,  143,   21, 7143,   21, 5767,   22, 5793,   21,   16,   47,
         836,  630,  691,   21, 3547, 2896,   22,   21,    3,    2]))


In [ ]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    encoder_vocab_size: int
    decoder_vocab_size: int
    demb: int
    h: int
    N_encoder: int
    N_decoder: int
    max_seq_len: int
    dropout: float



class Transformer(nn.Module):
    def __init__(self, encoder, decoder, config):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.proj = nn.Linear(config.demb, config.decoder_vocab_size)

    def forward(self, src, src_mask, trg, trg_pad_mask):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(trg, enc_output, src_mask, trg_pad_mask)
        return self.proj(dec_output)
    
def make_model(config):
    model = Transformer(Encoder(config), Decoder(config), config).to(DEVICE)

    # initialize model parameters
    # it seems that this initialization is very important!
    for p in model.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    return model

batch_size = 128

class Dataloaders:
    'Dataloaders contains train_loader, test_loader and valid_loader for training and evaluation '
    def __init__(self):
        self.train_dataset = TranslationDataset(train_tokenized)
        self.valid_dataset = TranslationDataset(valid_tokenized)
        self.test_dataset  = TranslationDataset(test_tokenized)
        
        # each batch returned by dataloader will be padded such that all the texts in
        # that batch have the same length as the longest text in that batch
        self.train_loader = torch.utils.data.DataLoader(self.train_dataset, batch_size=batch_size,
                                                shuffle=True, collate_fn = pad_sequence)
        self.valid_loader = torch.utils.data.DataLoader(self.valid_dataset, batch_size=batch_size,
                                                shuffle=True, collate_fn = pad_sequence)
        self.test_loader = torch.utils.data.DataLoader(self.test_dataset, batch_size=batch_size,
                                                shuffle=True, collate_fn = pad_sequence)
        
dataloaders = Dataloaders()
train_loader = dataloaders.train_loader
train_loader.dataset[2]

import tqdm
num_batches = len(dataloaders.train_loader)
pbar = tqdm.tqdm(
        enumerate(dataloaders.train_loader),
        total=num_batches
    )




  0%|          | 1/227 [00:08<33:17,  8.84s/it]


In [ ]:
from tqdm import tqdm
import numpy as np
import torch

def make_batch_input(x, y):
    src = x.to(DEVICE)

    trg_in = y[:, :-1].to(DEVICE)

    trg_out = y[:, 1:].contiguous().view(-1).to(DEVICE)

    src_pad_mask = (src == PAD).view(src.size(0), 1, 1, src.size(-1))

    trg_pad_mask = (trg_in == PAD).view(
        trg_in.size(0), 1, 1, trg_in.size(-1)
    )

    return src, trg_in, trg_out, src_pad_mask, trg_pad_mask


def train_epoch(model, dataloaders, optim, scheduler, loss_fn):
    model.train()

    grad_norm_clip = 1.0
    losses = []

    num_batches = len(dataloaders.train_loader)

    pbar = tqdm(
        enumerate(dataloaders.train_loader),
        total=num_batches
    )

    for idx, (x, y) in pbar:
        optim.zero_grad()

        src, trg_in, trg_out, src_pad_mask, trg_pad_mask = make_batch_input(x, y)

        pred = model.forward(src, src_pad_mask, trg_in, trg_pad_mask)

        # check for NaNs in model outputs
        if torch.isnan(pred).any():
            print("NaN detected in model outputs at batch", idx)
            print("src dtype/device:", src.dtype, src.device)
            for name, p in model.named_parameters():
                if torch.isnan(p).any():
                    print("NaN in parameter:", name)
                    break
            raise RuntimeError("NaN in model outputs")

        pred = pred.view(-1, pred.size(-1))

        loss = loss_fn(pred, trg_out)

        # Diagnostic: check for NaN/Inf loss
        if torch.isnan(loss) or torch.isinf(loss):
            print("NaN/Inf loss at batch", idx)
            torch.autograd.set_detect_anomaly(True)
            raise RuntimeError("NaN/Inf loss")

        loss.backward()

        # Diagnostic: check for NaNs in grads
        for name, p in model.named_parameters():
            if p.grad is not None and torch.isnan(p.grad).any():
                print("NaN in gradient for parameter:", name)
                raise RuntimeError("NaN in gradients")

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            grad_norm_clip
        )

        optim.step()
        scheduler.step()

        losses.append(loss.item())

        if idx > 0 and idx % 50 == 0:
            pbar.set_description(
                f"train loss={loss.item():.3f}, "
                f"lr={scheduler.get_last_lr()[0]:.5f}"
            )

    return np.mean(losses)


def validate(model, dataloader):
    """
    Compute the validation loss.
    """

    model.eval()
    losses = []

    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

    with torch.no_grad():
        for x, y in dataloader:
            src, trg_in, trg_out, src_pad_mask, trg_pad_mask = make_batch_input(x, y)

            pred = model.forward(src, src_pad_mask, trg_in, trg_pad_mask)

            pred = pred.view(-1, pred.size(-1))

            loss = loss_fn(pred, trg_out)

            losses.append(loss.item())

    return np.mean(losses)

def train(model, dataloaders, epochs, optim, scheduler, warmup_steps, loss_fn):
    global early_stop_count

    best_valid_loss = float("inf")

    for ep in range(epochs):
        train_loss = train_epoch(model, dataloaders, optim, scheduler, loss_fn)

        valid_loss = validate(model, dataloaders.valid_loader)

        print(
            f"ep: {ep}: "
            f"train_loss={train_loss:.5f}, "
            f"valid_loss={valid_loss:.5f}"
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss

        else:
            if scheduler.last_epoch > 2 * warmup_steps:
                early_stop_count -= 1

                if early_stop_count <= 0:
                    return train_loss, valid_loss

    return train_loss, valid_loss

In [62]:

import sacrebleu
def translate(model, x):
    'translate source sentences into the target language, without looking at the answer'
    with torch.no_grad():
        dB = x.size(0)
        y = torch.tensor([[BOS]*dB], dtype=torch.long, device=DEVICE).view(dB, 1)
        x_pad_mask = (x == PAD).view(x.size(0), 1, 1, x.size(-1))
        memory = model.encoder(x, x_pad_mask)
        for i in range(max_seq_len):
            y_pad_mask = (y == PAD).view(y.size(0), 1, 1, y.size(-1))
            logits = model.decoder(y, memory, x_pad_mask, y_pad_mask)
            last_output = logits.argmax(-1)[:, -1]
            last_output = last_output.view(dB, 1)
            y = torch.cat((y, last_output), 1)
    return y

def remove_pad(sent):
    '''truncate the sentence if BOS is in it,
     otherwise simply remove the padding tokens at the end'''
    if sent.count(EOS)>0:
      sent = sent[0:sent.index(EOS)+1]
    while sent and sent[-1] == PAD:
            sent = sent[:-1]
    return sent

def decode_sentence(detokenizer, sentence_ids):
    'convert a tokenized sentence (a list of numbers) to a literal string'
    if not isinstance(sentence_ids, list):
        sentence_ids = sentence_ids.tolist()
    sentence_ids = remove_pad(sentence_ids)
    return detokenizer(sentence_ids).replace("", "")\
           .replace("", "").strip().replace(" .", ".")

def evaluate(model, dataloader, num_batch=None):
    'evaluate the model, and compute the BLEU score'
    model.eval()
    refs, cans, bleus = [], [], []
    with torch.no_grad():
        for idx, (x, y) in enumerate(dataloader):
            src, trg_in, trg_out, src_pad_mask, trg_pad_mask = make_batch_input(x,y)
            translation = translate(model, src)
            trg_out = trg_out.view(x.size(0), -1)
            refs = refs + [decode_sentence(detokenizers["de"], trg_out[i]) for i in range(len(src))]
            cans = cans + [decode_sentence(detokenizers["de"], translation[i]) for i in range(len(src))] 
            if num_batch and idx>=num_batch:
                break
        print(min([len(x) for x in refs]))
        bleus.append(sacrebleu.corpus_bleu(cans, [refs]).score)
        # print some examples
        for i in range(3):
            print(f'src:  {decode_sentence(detokenizers["en"], src[i])}')
            print(f'trg:  {decode_sentence(detokenizers["en"], trg_out[i])}')
            print(f'pred: {decode_sentence(detokenizers["en"], translation[i])}')
        return np.mean(bleus)

In [64]:
config = ModelConfig(encoder_vocab_size = vocab_sizes["en"], 
                     decoder_vocab_size=vocab_sizes["de"],
                     demb=512,
                     h=8,
                     N_encoder=3, 
                     N_decoder=3, 
                     max_seq_len=max_seq_len,
                     dropout=0.1
                     )


def make_lr_fn(warmup_steps, demb):
    """Return a callable lr_lambda(step) for LambdaLR using the """
    """Transformer warmup schedule: demb^{-0.5} * min((step+1)^{-0.5}, (step+1)*warmup^{-1.5})"""
    def lr_lambda(step: int) -> float:
        step = max(step, 0)
        return demb ** (-0.5) * min((step + 1) ** (-0.5), (step + 1) * (warmup_steps ** -1.5))
    return lr_lambda


data_loaders = Dataloaders()
train_size = len(data_loaders.train_loader)*batch_size
model = make_model(config)
model_size = sum([p.numel() for p in model.parameters()])
print(f'model_size: {model_size}, train_set_size: {train_size}')
warmup_steps = 3*len(data_loaders.train_loader)
# lr first increases in the warmup steps, and then decreases
lr_fn = make_lr_fn(warmup_steps, config.demb)
# Use a stable initial optimizer LR to avoid exploding gradients
optimizer = torch.optim.Adam(model.parameters(), lr=0.5, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_fn)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
early_stop_count = 2
train_loss, valid_loss = train(model, data_loaders, epochs=7, optim=optimizer, scheduler=scheduler, warmup_steps=warmup_steps, loss_fn=loss_fn)
test_loss  = validate(model, data_loaders.test_loader)

model_size: 36558604, train_set_size: 29056


train loss=3.061, lr=0.00015: 100%|██████████| 227/227 [09:03<00:00,  2.40s/it]


ep: 0: train_loss=5.16605, valid_loss=2.78865


train loss=2.517, lr=0.00032: 100%|██████████| 227/227 [08:52<00:00,  2.34s/it]


ep: 1: train_loss=2.60520, valid_loss=2.46146


...

train loss=2.211, lr=0.00030: 100%|██████████| 227/227 [10:21<00:00,  2.74s/it]


ep: 8: train_loss=2.13023, valid_loss=2.21907


train loss=2.088, lr=0.00028: 100%|██████████| 227/227 [10:19<00:00,  2.73s/it]


ep: 9: train_loss=2.09444, valid_loss=2.21842
train set examples:
22
src:  ein hellbrauner hund l ⁇ uft vor einem verschwommenen hintergrund durch das gras.


IndexError: Out of range: piece id is out of range.

In [65]:
print("train set examples:")
train_bleu = evaluate(model, data_loaders.train_loader, 10)
print("validation set examples:")
valid_bleu = evaluate(model, data_loaders.valid_loader)
print("test set examples:")
test_bleu  = evaluate(model, data_loaders.test_loader)
print(f'train_loss: {train_loss:.4f}, valid_loss: {valid_loss:.4f}, test_loss: {test_loss:.4f}')
print(f'test_bleu: {test_bleu:.4f}, valid_bleu: {valid_bleu:.4f} train_bleu: {train_bleu:.4f}')

train set examples:
21
src:  ein schlafendes kleines kind h ⁇ lt ein pl ⁇ schtier.
trg:  break Two at in waterway Two new region shoes Two at luggage hertting at sleeve specta luggage mall Police shoes du break pond shoelacesacross her shoes white items in wetsuit Two a
pred: beach standing ice ice ice ice ice ice ice ice ice house house house house cellphone ice ice ice ice ice standing ice ice someone house bar someone someone board house walks someone board ice walks walks walks walks walks walks walks walks ice walks ice ice ice ice walks
src:  zwei junge , blonde br ⁇ der haben ihre arme einer hinter dem anderen inmitten einer wiese ausgebreitet.
trg:  Two over mov adult Two oval fence Two is taste Twoidewalk antique at africa mannequin Two crash snowball Two lights in atrk blouse register Two is Two winning vest Hug shoes Two crash Two motorcyclists antiquey items shoes speak breakN bullet shoes cigarettes bearer
pred: beach standing standing ice ice standing hard hard hard hard 

IndexError: Out of range: piece id is out of range.